# Fine-tune pretrained Transformer cho intent tiếng Việt: PhoBERT, BERT, RoBERTa

**Mục tiêu:** thay LLM gọi API (Gemini / gpt-oss) bằng một model **tự host** chạy trên CPU, nhanh và không tốn phí mỗi lần gọi.

**Dữ liệu**
- `dataset.jsonl`: 1000 câu, 14 nhãn, chủ yếu tiếng Việt với nhiều kiểu viết (teencode, không dấu, giọng miền, tiếng Anh). Đây là tập train/test chính.
- *(tuỳ chọn)* Bitext tiếng Anh từ HuggingFace, gộp nhãn về 14 nhãn của ta, **chỉ thêm vào tập train**.
- *(tuỳ chọn)* `benchmark_scenarios.json`: 100 kịch bản của repo, dùng làm tập test phụ.

**Kết quả xuất ra** (`/kaggle/working/`)
- `leaderboard.csv`: so sánh các model về độ chính xác, Macro-F1, mức độ tự tin đúng mực (ECE), tỷ lệ câu được tự động xử lý, độ trễ CPU.
- `intent_artifact/multi/`: model tốt nhất dạng **ONNX int8** kèm `meta.json`, đúng định dạng `OnnxTransformerBackend` của `services/intent_service/src/predictor.py` (nhánh `refine`).

**Thiết lập Kaggle:** Accelerator **GPU T4 x2** (tự train 2 model song song; T4 x1 hoặc P100 vẫn chạy được, chỉ chậm hơn), **Internet On**. Dữ liệu tự lấy bằng cách clone repo (nhánh `REPO_BRANCH`), hoặc upload `dataset.jsonl` qua *Add Data*.

## 0. PhoBERT, BERT, RoBERTa là gì? Chọn cái nào?

Cả ba đều là **Transformer encoder đã được pretrain** trên lượng văn bản rất lớn, bằng cách che bớt một số từ rồi bắt model đoán lại (masked language modeling). Sau bước đó, model đã "hiểu" ngôn ngữ. Ta chỉ cần gắn thêm một **lớp phân loại** lên vector đại diện cho cả câu (token `[CLS]` / `<s>`) rồi **fine-tune** toàn bộ trên ~800 câu có nhãn. Đây là transfer learning: dữ liệu ít vẫn học tốt.

| Model | Pretrain trên | Tham số | Ghi chú cho bài toán này |
|---|---|---|---|
| **BERT** (`bert-base-multilingual-cased`, mBERT) | Wikipedia của 104 ngôn ngữ | 178M | Mốc so sánh gốc (Google, 2018). Tiếng Việt chỉ là một phần nhỏ trong dữ liệu, nên thường kém hơn PhoBERT và XLM-R |
| **RoBERTa** (`roberta-base`) | 160GB văn bản **tiếng Anh** | 125M | Là BERT được train kỹ hơn: nhiều dữ liệu hơn, bỏ tác vụ đoán câu tiếp theo (NSP), che từ thay đổi theo từng lượt. **Không biết tiếng Việt** nên chỉ đáng dùng cho câu tiếng Anh |
| **XLM-RoBERTa** (`xlm-roberta-base`) | 2.5TB CommonCrawl, 100 ngôn ngữ (có tiếng Việt khá nhiều) | 278M | RoBERTa **đa ngôn ngữ**. Xử lý được câu trộn Việt–Anh, không dấu, teencode. **Ứng viên mạnh nếu dùng 1 model cho cả 2 ngôn ngữ** |
| **PhoBERT** (`vinai/phobert-base-v2`) | 20GB văn bản tiếng Việt + 120GB OSCAR tiếng Việt (VinAI) | 135M | RoBERTa **chuyên tiếng Việt**, thường mạnh nhất với tiếng Việt chuẩn. **Bắt buộc tách từ** (`học sinh` thành `học_sinh`, dùng pyvi). Có thể yếu hơn ở câu không dấu và tiếng Anh |
| *DistilBERT đa ngôn ngữ* (`distilbert-base-multilingual-cased`) | Chưng cất (distill) từ mBERT | 135M (6 layer) | Nhanh gần gấp đôi mBERT, phù hợp khi cần độ trễ thấp |

**Cách đọc kết quả:** với tập test 200 câu, mỗi câu sai làm giảm accuracy 0.5%. Hai model chênh nhau dưới 1–2% coi như ngang nhau. Hãy bật `SEEDS = [42, 7, 2024]` để xem độ dao động giữa các lần train.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyvi", "onnx", "onnxruntime", "datasets"], check=False)

In [ ]:
import gc, glob, json, os, random, re, time, unicodedata
from pathlib import Path
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU = torch.cuda.device_count()
GPUS = [f"cuda:{i}" for i in range(N_GPU)] or ["cpu"]
print("devices:", GPUS, [torch.cuda.get_device_name(i) for i in range(N_GPU)])
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
WORK.mkdir(parents=True, exist_ok=True)

def find(pattern):
    return sorted(set(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True) + glob.glob(pattern)
                      + glob.glob(f"data/intent_eval/{pattern}") + glob.glob(f"data/{pattern}")))

### Lấy dữ liệu

Notebook tìm `dataset.jsonl` theo thứ tự:
1. File đã upload vào `/kaggle/input/` (*Add Data*).
2. Thư mục repo hiện tại (khi chạy local).
3. Nếu chưa thấy ở hai chỗ trên thì **clone repo GitHub**, nhánh `REPO_BRANCH`, rồi lấy `data/intent_eval/dataset.jsonl` từ đó.

Cách 3 cần bật Internet và cần dataset đã được push lên nhánh đó.

In [ ]:
REPO_URL = "https://github.com/DunnTapCodee/E-commerce-Support-Copilot.git"
REPO_BRANCH = "hung"      # nhánh chứa data/intent_eval/dataset.jsonl

if not find("dataset.jsonl"):
    repo = WORK / "repo"
    if (repo / ".git").exists():   # đã clone ở lần chạy trước: cập nhật lên bản mới nhất
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo), "reset", "--hard", "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(repo)], check=True)
    os.chdir(repo)
    print("repo commit:", subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())

print("dataset:", find("dataset.jsonl"))
print("benchmark:", find("benchmark_scenarios.json"))
assert find("dataset.jsonl"), "Không tìm thấy dataset.jsonl: upload qua Add Data hoặc kiểm tra REPO_BRANCH đã có file."

## 1. Cấu hình

In [ ]:
MODELS = {
    # tên ngắn: (HF id, cần tách từ tiếng Việt?)
    "phobert":   ("vinai/phobert-base-v2", True),
    "xlmr":      ("xlm-roberta-base", False),
    "mbert":     ("bert-base-multilingual-cased", False),
    "distil_mbert": ("distilbert-base-multilingual-cased", False),
    "roberta_en": ("roberta-base", False),   # đối chứng: không biết tiếng Việt
}
RUN = ["phobert", "xlmr", "mbert", "distil_mbert", "roberta_en"]   # bỏ bớt để chạy nhanh
SEEDS = [42]            # [42, 7, 2024] để đo độ dao động (thời gian x3)

USE_BITEXT_EN = True    # thêm câu tiếng Anh từ Bitext vào train
EN_PER_LABEL = 60       # số câu EN tối đa mỗi nhãn (để EN không lấn át VI)

EPOCHS, LR, BATCH, MAX_LEN, WARMUP, WD = 10, 3e-5, 16, 96, 0.1, 0.01
PATIENCE = 3            # dừng sớm nếu val Macro-F1 không cải thiện sau 3 epoch
TARGET_ACCEPTED_ACC = 0.95   # ngưỡng confidence: câu được giữ phải đúng ≥ 95%

LABELS = ["track_order", "cancel_order", "delivery_time", "delivery_issue", "shipping_costs",
          "return_policy", "return_product", "refund_policy", "request_refund", "refund_status",
          "payment_issue", "human_agent", "other_ecommerce", "out_of_scope"]
L2I = {l: i for i, l in enumerate(LABELS)}

## 2. Dữ liệu và chia tập

In [ ]:
df = pd.read_json(find("dataset.jsonl")[0], lines=True)
assert set(df.intent) <= set(LABELS), set(df.intent) - set(LABELS)
df["src"] = "vi_handwritten"

# Test 20% cố định (giống notebook TF-IDF, cùng random_state) để so sánh được với ML cổ điển và LLM.
trainval, test_df = train_test_split(df, test_size=0.2, stratify=df.intent, random_state=42)
train_df, val_df = train_test_split(trainval, test_size=0.125, stratify=trainval.intent, random_state=42)  # 70/10/20
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")

### (Tuỳ chọn) Thêm Bitext tiếng Anh vào train

Bitext có 46 intent. Bảng `BITEXT_MAP` gộp chúng về 14 nhãn của ta. Các intent mơ hồ như `wrong_item`, `missing_item`, `damaged_delivery`, `product_issue`, `pay` bị **bỏ** vì có thể thuộc nhiều nhãn khác nhau. Placeholder dạng `{{Order Number}}` được điền giá trị mẫu.

In [ ]:
BITEXT_MAP = {
    "track_order": "track_order", "track_delivery": "track_order", "cancel_order": "cancel_order",
    "delivery_time": "delivery_time", "delivery_issue": "delivery_issue", "shipping_costs": "shipping_costs",
    "return_policy": "return_policy", "return_product": "return_product", "return_product_online": "return_product",
    "return_product_in_store": "return_product", "refund_policy": "refund_policy", "request_refund": "request_refund",
    "refund_status": "refund_status", "payment_issue": "payment_issue", "human_agent": "human_agent",
    "customer_service": "human_agent",
    **{k: "other_ecommerce" for k in ["add_product", "availability", "availability_in_store", "availability_online",
        "change_account", "change_order", "close_account", "open_account", "order_history", "payment_methods",
        "product_information", "recover_password", "remove_product", "request_invoice", "sales_period",
        "store_location", "store_opening_hours", "submit_feedback", "submit_product_feedback",
        "submit_product_idea", "use_app", "technical_issue", "exchange_product", "exchange_product_in_store",
        "request_right_to_rectification"]},
}
FILL = {"order number": "ORD-48213", "person name": "Lan", "store location": "Quận 7",
        "product name": "tai nghe", "refund amount": "350k"}

def fill_placeholders(t):
    return re.sub(r"\{\{\s*([^}]+?)\s*\}\}", lambda m: FILL.get(m.group(1).lower(), m.group(1)), t)

en_df = pd.DataFrame()
if USE_BITEXT_EN:
    from datasets import load_dataset
    bx = load_dataset("bitext/Bitext-retail-ecommerce-llm-chatbot-training-dataset", split="train").to_pandas()
    text_col = next(c for c in ["instruction", "utterance", "text"] if c in bx.columns)
    bx = bx.assign(intent=bx["intent"].map(BITEXT_MAP)).dropna(subset=["intent"])
    bx["text"] = bx[text_col].map(fill_placeholders)
    en_df = (bx.drop_duplicates("text").groupby("intent", group_keys=False)
               .apply(lambda g: g.sample(min(len(g), EN_PER_LABEL), random_state=42)))[["text", "intent"]]
    en_df["src"] = "bitext_en"
    print("Bitext EN thêm vào train:", len(en_df)); display(en_df.intent.value_counts())

train_full = pd.concat([train_df, en_df], ignore_index=True)
print("train cuối:", len(train_full))

In [ ]:
# Tập test phụ: 100 kịch bản benchmark của repo (nhãn warranty gộp vào other_ecommerce).
bench_df = pd.DataFrame()
paths = find("benchmark_scenarios.json")
if paths:
    b = pd.DataFrame(json.load(open(paths[0], encoding="utf-8")))
    bench_df = pd.DataFrame({"text": b["query"], "intent": b["expected_intent"].replace({"warranty": "other_ecommerce"})})
    bench_df = bench_df[bench_df.intent.isin(LABELS)].drop_duplicates("text")
    print("benchmark:", len(bench_df), "câu (đã bỏ trùng)")

## 3. Tiền xử lý: giống hệt lúc serving

`predictor.py` (nhánh `refine`) làm 2 bước: `preprocess()` (chuẩn hoá NFC, gom khoảng trắng, hạ chữ thường), rồi nếu `needs_word_segmentation` thì tách từ bằng pyvi. Notebook làm **đúng y như vậy** để kết quả train khớp với lúc chạy thật.

Lưu ý: hạ chữ thường làm mất thông tin hoa/thường mà mBERT-*cased* đã học, nhưng ta vẫn giữ để khớp serving.

In [ ]:
from pyvi import ViTokenizer

def preprocess(t):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(t))).strip().lower()

def prepare(t, segment):
    t = preprocess(t)
    return ViTokenizer.tokenize(t) if segment else t

print(prepare("Đơn hàng ORD-1001 của tôi đang ở đâu vậy shop?", True))

## 4. Vòng train (PyTorch thuần, fp16 khi có GPU)

Mọi hàm nhận tham số `dev` (`cuda:0`, `cuda:1`, `cpu`) để có thể chạy **nhiều model song song, mỗi GPU một model** (xem mục 6).

In [ ]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

SEED_LOCK = __import__("threading").Lock()   # khởi tạo model dưới khoá để seed không bị luồng khác chen vào

def encode(tok, texts, segment):
    return tok([prepare(t, segment) for t in texts], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")

def batches(tok, frame, segment, bs, dev, shuffle=False, seed=0):
    idx = np.arange(len(frame))
    if shuffle: np.random.default_rng(seed).shuffle(idx)
    texts, ys = frame.text.to_numpy(), frame.intent.map(L2I).to_numpy()
    for i in range(0, len(idx), bs):
        j = idx[i:i + bs]
        enc = encode(tok, texts[j], segment)
        enc["labels"] = torch.tensor(ys[j])
        yield {k: v.to(dev) for k, v in enc.items()}

@torch.no_grad()
def logits_of(model, tok, frame, segment, dev):
    model.eval(); out = []
    for b in batches(tok, frame, segment, 64, dev):
        b.pop("labels")
        with torch.autocast("cuda" if dev.startswith("cuda") else "cpu", enabled=dev.startswith("cuda")):
            out.append(model(**b).logits.float().cpu())
    return torch.cat(out).numpy()

def train_one(key, seed, dev):
    hf_id, segment = MODELS[key]
    gpu = dev.startswith("cuda")
    with SEED_LOCK:
        set_seed(seed)
        tok = AutoTokenizer.from_pretrained(hf_id)
        model = AutoModelForSequenceClassification.from_pretrained(
            hf_id, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=L2I)
    model.to(dev)
    steps = EPOCHS * ((len(train_full) + BATCH - 1) // BATCH)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=gpu)
    best, best_state, bad, hist = -1, None, 0, []
    t0 = time.time()
    for ep in range(EPOCHS):
        model.train(); tot = n = 0
        for b in batches(tok, train_full, segment, BATCH, dev, shuffle=True, seed=seed + ep):
            with torch.autocast("cuda" if gpu else "cpu", enabled=gpu):
                loss = model(**b).loss
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
            tot += loss.item(); n += 1
        vf1 = f1_score(val_df.intent.map(L2I), logits_of(model, tok, val_df, segment, dev).argmax(1), average="macro")
        hist.append({"epoch": ep + 1, "loss": tot / n, "val_f1": vf1})
        print(f"  [{dev} {key} s{seed}] epoch {ep+1}: loss {tot/n:.3f}  val macro-F1 {vf1:.4f}")
        if vf1 > best:
            best, bad = vf1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE: break
    model.load_state_dict(best_state)
    return model, tok, time.time() - t0, hist

## 5. Hiệu chỉnh confidence và chọn ngưỡng

- **Temperature scaling:** tìm hệ số `T` sao cho `softmax(logits / T)` sát với xác suất đúng thật, tối ưu trên **tập val**. Gateway chặn theo confidence nên con số này phải đáng tin.
- **ECE** (Expected Calibration Error): độ lệch trung bình giữa confidence model báo và tỷ lệ đúng thực tế. Càng thấp càng tốt.
- **Ngưỡng:** confidence nhỏ nhất sao cho các câu được giữ lại đúng ≥ 95% trên val. Câu dưới ngưỡng chuyển cho nhân viên. **Coverage** là tỷ lệ câu được model tự xử lý. Tập test **không** được dùng để chọn ngưỡng.

In [ ]:
def ece(y, p, bins=15):
    conf, correct, tot = p.max(1), p.argmax(1) == y, 0.0
    for i in range(bins):
        m = (conf > i / bins) & (conf <= (i + 1) / bins)
        if m.any(): tot += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(tot)

def fit_temperature(logits, y):
    nll = lambda T: -np.log(np.clip(softmax(logits / T, 1)[np.arange(len(y)), y], 1e-12, 1)).mean()
    return float(minimize_scalar(nll, bounds=(0.05, 10), method="bounded").x)

def pick_threshold(p, y):
    conf, ok = p.max(1), p.argmax(1) == y
    for t in np.round(np.arange(0.30, 0.96, 0.01), 2):
        keep = conf >= t
        if keep.any() and ok[keep].mean() >= TARGET_ACCEPTED_ACC: return float(t)
    return None

def cpu_latency_ms(model, tok, segment, n=50):
    # đo SAU khi train xong và chạy tuần tự, để 2 luồng train không tranh CPU làm sai số đo
    m = model.to("cpu").float().eval(); sample = test_df.text.sample(n, random_state=0).tolist()
    with torch.no_grad():
        m(**encode(tok, sample[:1], segment))   # warmup
        t0 = time.perf_counter()
        for s in sample: m(**encode(tok, [s], segment))
    return (time.perf_counter() - t0) / n * 1000

## 6. Chạy so sánh, song song trên nhiều GPU

Mỗi cặp (model, seed) là một job trong hàng đợi. Mỗi GPU có một luồng riêng: lấy job, train, lấy job tiếp. GPU nào xong trước thì nhận việc tiếp, nên tải tự cân bằng.
- **2×T4:** 2 model train cùng lúc, tổng thời gian giảm gần một nửa so với 1 GPU.
- **1 GPU hoặc CPU:** chạy tuần tự như bình thường.

Log của 2 GPU in xen kẽ nhau, mỗi dòng có tiền tố `[cuda:0 …]` hoặc `[cuda:1 …]` để phân biệt.

Lưu ý: khi chạy song song, kết quả không lặp lại y hệt đến từng chữ số giữa các lần chạy (do thứ tự thực thi trên GPU). Chênh lệch nhỏ hơn nhiều so với dao động giữa các seed.

In [ ]:
import queue, threading, traceback
y_val, y_test = val_df.intent.map(L2I).to_numpy(), test_df.intent.map(L2I).to_numpy()
jobs = queue.Queue()
# model lớn chạy trước để 2 GPU kết thúc gần cùng lúc
order = sorted(RUN, key=lambda k: -{"xlmr": 278, "mbert": 178, "phobert": 135, "distil_mbert": 135, "roberta_en": 125}.get(k, 150))
for key in order:
    for seed in SEEDS: jobs.put((key, seed))

rows, store, errors, LOCK = [], {}, [], threading.Lock()

def evaluate_job(key, seed, dev):
    model, tok, train_s, hist = train_one(key, seed, dev)
    seg = MODELS[key][1]
    val_logits = logits_of(model, tok, val_df, seg, dev)
    T = fit_temperature(val_logits, y_val)
    thr = pick_threshold(softmax(val_logits / T, 1), y_val)
    p_test = softmax(logits_of(model, tok, test_df, seg, dev) / T, 1)
    pred = p_test.argmax(1); keep = p_test.max(1) >= (thr if thr is not None else 1.01)
    r = {"model": key, "seed": seed, "gpu": dev, "test_acc": accuracy_score(y_test, pred),
         "test_macro_f1": f1_score(y_test, pred, average="macro"), "ece": ece(y_test, p_test),
         "temperature": T, "threshold": thr, "coverage": keep.mean(),
         "acc_on_kept": (pred[keep] == y_test[keep]).mean() if keep.any() else np.nan,
         "epochs": len(hist), "train_s": train_s, "params_M": sum(p.numel() for p in model.parameters()) / 1e6}
    if len(bench_df):
        r["bench_acc"] = accuracy_score(bench_df.intent.map(L2I),
                                        (logits_of(model, tok, bench_df, seg, dev) / T).argmax(1))
    vf1 = max(h["val_f1"] for h in hist)
    state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    del model; gc.collect(); torch.cuda.empty_cache()
    with LOCK:
        rows.append(r)
        if key not in store or vf1 > store[key]["val_f1"]:
            store[key] = {"val_f1": vf1, "T": T, "thr": thr, "p_test": p_test, "state": state}
    print(f"[{dev}] DONE {key} s{seed}: acc {r['test_acc']:.4f} | macro-F1 {r['test_macro_f1']:.4f} | {train_s/60:.1f} phút")

def worker(dev):
    while True:
        try: key, seed = jobs.get_nowait()
        except queue.Empty: return
        try: evaluate_job(key, seed, dev)
        except Exception as e:   # một model lỗi (ví dụ hết VRAM) không làm dừng các model khác
            with LOCK: errors.append((key, seed, dev, repr(e)))
            traceback.print_exc(); torch.cuda.empty_cache()

t0 = time.time()
threads = [threading.Thread(target=worker, args=(d,)) for d in GPUS]
for th in threads: th.start()
for th in threads: th.join()
print(f"Tổng thời gian: {(time.time() - t0) / 60:.1f} phút trên {GPUS}")
if errors: print("LỖI:", errors)

In [ ]:
# Đo độ trễ CPU tuần tự cho từng model (mỗi model đo 1 lần, dùng bản tốt nhất)
lat = {}
for key in store:
    hf_id, seg = MODELS[key]
    tok = AutoTokenizer.from_pretrained(hf_id)
    m = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=len(LABELS))
    m.load_state_dict(store[key]["state"]); lat[key] = cpu_latency_ms(m, tok, seg); del m
    print(f"{key:14s} {lat[key]:.1f} ms/câu (PyTorch fp32, CPU)")
res = pd.DataFrame(rows); res["cpu_ms"] = res.model.map(lat)

In [ ]:
agg = res.groupby("model").agg(
    test_acc=("test_acc", "mean"), acc_std=("test_acc", "std"), test_macro_f1=("test_macro_f1", "mean"),
    ece=("ece", "mean"), coverage=("coverage", "mean"), acc_on_kept=("acc_on_kept", "mean"),
    cpu_ms=("cpu_ms", "mean"), params_M=("params_M", "first"), train_s=("train_s", "mean"),
    **({"bench_acc": ("bench_acc", "mean")} if "bench_acc" in res else {})
).sort_values("test_macro_f1", ascending=False)

# Thêm LLM nếu đã có kết quả (từ eval_llm.py) trên cùng tập test
for path in find("results_*.jsonl"):
    llm = pd.read_json(path, lines=True).set_index("id")["pred"]
    m = test_df.join(llm.rename("p"), on="id").dropna(subset=["p"])
    if len(m):
        agg.loc["LLM:" + Path(path).stem[8:], ["test_acc", "test_macro_f1"]] = [
            accuracy_score(m.intent, m.p), f1_score(m.intent, m.p, average="macro")]
agg.to_csv(WORK / "leaderboard.csv"); res.to_csv(WORK / "runs.csv", index=False)
agg.round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
a = agg.dropna(subset=["cpu_ms"])
ax[0].barh(agg.index, agg.test_macro_f1); ax[0].set_xlim(max(0, agg.test_macro_f1.min() - .1), 1); ax[0].set_title("Test Macro-F1")
ax[1].scatter(a.cpu_ms, a.test_macro_f1)
for k, r in a.iterrows(): ax[1].annotate(k, (r.cpu_ms, r.test_macro_f1))
ax[1].set_xlabel("CPU latency / câu (ms, PyTorch fp32)"); ax[1].set_ylabel("Macro-F1"); ax[1].set_title("Độ chính xác và tốc độ")
plt.tight_layout(); plt.show()

## 7. Phân tích lỗi của model tốt nhất

In [ ]:
BEST = agg.dropna(subset=["cpu_ms"]).index[0]     # hoặc đặt tay, ví dụ "xlmr"
s = store[BEST]; pred = s["p_test"].argmax(1); conf = s["p_test"].max(1)
print("Best:", BEST, "| T =", round(s["T"], 3), "| threshold =", s["thr"])
print(classification_report(y_test, pred, labels=range(len(LABELS)), target_names=LABELS, zero_division=0))

cm = confusion_matrix(y_test, pred, labels=range(len(LABELS)))
fig, ax = plt.subplots(figsize=(9, 8)); ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(LABELS)), LABELS, rotation=75); ax.set_yticks(range(len(LABELS)), LABELS)
for i, j in zip(*np.nonzero(cm)): ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
plt.tight_layout(); plt.show()

t = test_df.assign(pred=[LABELS[i] for i in pred], conf=conf)
t["ok"] = t.pred == t.intent
display(t.groupby("style").ok.agg(["mean", "count"]).query("count >= 3").sort_values("mean").head(12))
display(t.loc[~t.ok, ["text", "intent", "pred", "conf"]].sort_values("conf", ascending=False))

## 8. Export ONNX int8 cho `intent_service`

Notebook xuất model rồi lượng tử hoá động (dynamic quantization) xuống int8. Model chạy trên CPU bằng `onnxruntime`, nhỏ đi khoảng 4 lần và nhanh hơn PyTorch.

Thư mục xuất ra có dạng:
```
intent_artifact/
  router/        meta.json + vi_notone.txt   (predictor yêu cầu có router)
  multi/         model_int8.onnx + tokenizer + meta.json
```
Cách dùng: copy `intent_artifact/*` vào `artifacts/models/intent/`, rồi đặt `INTENT_ROUTING_MODE=multilingual` và `INTENT_BACKEND=ml` (hoặc `hybrid`).

Lưu ý: `predictor.py` coi những nhãn không có trong `TARGET_INTENTS` là ngoài phạm vi, nên `other_ecommerce` và `out_of_scope` sẽ được chuyển cho nhân viên.

In [ ]:
EXPORT = BEST
hf_id, seg = MODELS[EXPORT]
tok = AutoTokenizer.from_pretrained(hf_id)
model = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=L2I)
model.load_state_dict(store[EXPORT]["state"]); model.eval().float()

out = WORK / "intent_artifact" / "multi"; out.mkdir(parents=True, exist_ok=True)
dummy = tok([prepare("đơn hàng của tôi đâu rồi", seg)], return_tensors="pt")
inputs = [k for k in ["input_ids", "attention_mask", "token_type_ids"] if k in dummy]
import inspect
extra = {"dynamo": False} if "dynamo" in inspect.signature(torch.onnx.export).parameters else {}  # torch mới mặc định dùng dynamo
torch.onnx.export(model, tuple(dummy[k] for k in inputs), str(out / "model.onnx"), input_names=inputs,
                  output_names=["logits"], opset_version=17,
                  dynamic_axes={**{k: {0: "batch", 1: "seq"} for k in inputs}, "logits": {0: "batch"}}, **extra)
from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic(str(out / "model.onnx"), str(out / "model_int8.onnx"), weight_type=QuantType.QInt8)
(out / "model.onnx").unlink()     # chỉ giữ bản int8
tok.save_pretrained(out)

In [ ]:
# Kiểm tra: ONNX int8 phải cho kết quả gần như PyTorch, và đo tốc độ thật trên CPU
import onnxruntime as ort
sess = ort.InferenceSession(str(out / "model_int8.onnx"), providers=["CPUExecutionProvider"])
names = {i.name for i in sess.get_inputs()}
T = store[EXPORT]["T"]
def onnx_proba(text):
    enc = tok(prepare(text, seg), return_tensors="np", truncation=True, max_length=MAX_LEN)
    return softmax(sess.run(None, {k: enc[k].astype(np.int64) for k in names})[0][0] / T)

t0 = time.perf_counter(); p_onnx = np.stack([onnx_proba(x) for x in test_df.text]); dt = (time.perf_counter() - t0) / len(test_df) * 1000
agree = (p_onnx.argmax(1) == store[EXPORT]["p_test"].argmax(1)).mean()
onnx_acc = accuracy_score(y_test, p_onnx.argmax(1))
print(f"ONNX int8: {dt:.1f} ms/câu | trùng dự đoán với PyTorch {agree:.1%} | test acc {onnx_acc:.4f}")
print(f"Kích thước: {sum(f.stat().st_size for f in out.iterdir()) / 1e6:.0f} MB")

In [ ]:
import importlib.metadata as im
meta = {"backend": "onnx_transformer", "method": f"ft-{EXPORT}", "base_model": hf_id, "lang": "multi",
        "classes": LABELS, "max_length": MAX_LEN, "temperature": store[EXPORT]["T"],
        "threshold": store[EXPORT]["thr"], "needs_word_segmentation": seg, "model_file": "model_int8.onnx",
        "library_versions": {p: im.version(p) for p in ["transformers", "torch", "onnxruntime"]},
        "metrics": {"test_acc_onnx_int8": float(onnx_acc), "latency_ms_onnx_int8": float(dt),
                    **{k: float(v) for k, v in agg.loc[EXPORT].dropna().items()}},
        "train_data": {"vi_handwritten": int(len(train_df)), "bitext_en": int(len(en_df))}}
json.dump(meta, open(out / "meta.json", "w"), ensure_ascii=False, indent=2)

# Router tối thiểu: predictor yêu cầu thư mục router/ tồn tại. Ở chế độ multilingual, router chỉ dùng để gắn nhãn ngôn ngữ cho log.
r = WORK / "intent_artifact" / "router"; r.mkdir(parents=True, exist_ok=True)
(r / "vi_notone.txt").write_text("\n".join(sorted(set("toi muon huy don hang hoan tien phi bao nhieu giao chua khong minh nay giup voi duoc cho nhan vien chinh sach doi tra san pham kiem tinh trang shop ad oi".split()))))
json.dump({"threshold": 0.2}, open(r / "meta.json", "w"))

subprocess.run(["zip", "-qr", str(WORK / "intent_artifact.zip"), "intent_artifact"], cwd=WORK)
print(json.dumps(meta, ensure_ascii=False, indent=1)[:1500])

In [ ]:
for q in ["đơn ORD-1001 tới đâu r shop", "toi bi tru tien 2 lan ma don van loi", "cho gặp nhân viên thật đi",
          "where is my refund", "hôm nay trời đẹp ghê", "bao lâu thì nhận được hàng zậy"]:
    p = onnx_proba(q); i = p.argmax()
    flag = "" if store[EXPORT]["thr"] and p[i] >= store[EXPORT]["thr"] else "  -> dưới ngưỡng, chuyển nhân viên"
    print(f"{q:42s} {LABELS[i]:16s} {p[i]:.2f}{flag}")

## Bước tiếp theo
1. **Thêm dữ liệu thật** là cách tăng điểm hiệu quả nhất. Pipeline dịch Bitext sang tiếng Việt (k01/k02 ở nhánh `refine`) có thể nhân dữ liệu train lên nhiều lần. Sau đó chạy lại notebook này với `train_vi.parquet`.
2. Nếu model tự host đạt ngang LLM trên tập test, có thể chuyển `INTENT_BACKEND=ml`: 0 đồng mỗi lần gọi, độ trễ vài chục ms. Hoặc dùng `hybrid`: model tự host xử lý câu có confidence cao, chỉ gửi câu khó cho LLM.
3. Rà lại các câu sai ở mục 7. Một phần lỗi có thể do **nhãn gốc sai**.